# Simulation of Moon TOD using real scan data

normaliser l'effet de l'atomosphère !

T = T_lune * exp(-tau*Masse_air)

avec Masse_air = 1/sin(elevation) et tau épaisseur optique atmosphère

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%config InlineBackend.figure_format='retina'
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

%matplotlib ipympl


from datetime import datetime, timezone
import matplotlib.pyplot as plt
import numpy as np
from astropy.visualization import astropy_mpl_style, quantity_support
# plt.style.use(astropy_mpl_style)
quantity_support()
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord, EarthLocation, AltAz, get_body

# plt.rc('figure',figsize=(20,12))
plt.rc('figure',figsize=(10,7))
plt.rc('font',size=16)

from scipy.signal import medfilt
from scipy.interpolate import interp1d
import glob
import gc
import pickle

from qubicpack.qubicfp import qubicfp
from qubicpack.iv import ADU2I
import qubic.lib.Calibration.Qfiber as ft
import qubic
import healpy as hp
from qubicpack.utilities import Qubic_DataDir
from qubic.lib import Qdictionary, Qscene, Qinstrument, Qacquisition, Qsamplings
from qubic.lib.MapMaking.Qcg import pcg
from pyoperators import DiagonalOperator
from qubic.lib.Qfoldertools import do_gif
from pyoperators import MPI
from scipy.optimize import least_squares
from mpl_toolkits.axes_grid1 import make_axes_locatable

import corner
import emcee

from importlib import reload
import healpy as hp

import time_domain_tools as tdt

import pipeline_moon_functions as pmf
import pipeline_moon_plotting as pmp
import SimulateMoonTOD as simumoon



Below we create a theoretical synthbeam for a detector that would have the same l.o.s. as the instrument

In [ ]:
# Mandatory imports

from qubic.lib.Qdictionary import qubicDict
from qubic.lib.Instrument.Qinstrument import QubicMultibandInstrument
from qubic.lib.Qscene import QubicScene

plt.rcParams['figure.figsize'] = (8,4)

def print_keys(d, keys):
    for key in keys:
        print(' - {:25}: {}'.format(key, d[key]))
    print('\n')



### This function retrieves the peaks information as a function of frequency, for each of the MultiBandInstrument sub-bands
def get_peaks_configuration(d, doplot=False, idet=None, debug=False):
    if d['config'] == 'TD':
        Ndet = 248
    elif d['config'] == 'FI':
        Ndet = 992
    else:
        print('Wrong config in dict')
        return 0


    print_keys(d, ['config', 'instrument_type', 'synthbeam', 'use_synthbeam_file', 'synthbeam_fraction', 'synthbeam_kmax'])
    try:
        q_instrument = QubicMultibandInstrument(d)
        q_scene = QubicScene(d)
        print('done', config, instrument_type, 'nf_sub={}'.format(nf_sub))
    except:
        print('oups ! failed instanciating QubicMultibandInstrument()')
        return 0,0,0,0
    
    n_nus = len(q_instrument)
    nus = np.zeros(n_nus)
    dnus = np.zeros(n_nus)
    print()
    print('The instrument has {} sub-frequencies'.format(n_nus))
    for i in range(n_nus):
        nus[i] = q_instrument.subinstruments[i].d['filter_nu']/1e9
        dnus[i] = q_instrument.subinstruments[i].d['filter_relative_bandwidth']*q_instrument.subinstruments[i].d['filter_nu']/1e9
        print('- {0:}: nu = {1:7.2f} GHz ; bw = {2:7.2f}'.format(i, nus[i], dnus[i]))

        # this edit doesn't work anymore!! I don't know why
        q_instrument.subinstruments[i].detector.center[0] = np.array([0, 0, -0.3]) # we take a fake central detector because we only care about theta and phi from zenith
        print("detector center", q_instrument.subinstruments[i].detector.center[0])
        
    thetas = np.zeros((n_nus, Ndet, (2*d['synthbeam_kmax']+1)**2))
    phis = np.zeros((n_nus, Ndet, (2*d['synthbeam_kmax']+1)**2))
    vals = np.zeros((n_nus, Ndet, (2*d['synthbeam_kmax']+1)**2))
    for i in range(n_nus):
        thetas[i,:,:], phis[i,:,:], vals[i,:,:] = q_instrument.subinstruments[i]._peak_angles(q_scene, 
                                                    q_instrument.subinstruments[i].d['filter_nu'], 
                                                    np.full_like(q_instrument.subinstruments[i].detector.center, np.array([0, 0, -0.3])), 
                                                    q_instrument.subinstruments[i].synthbeam, 
                                                    q_instrument.subinstruments[i].horn, 
                                                    q_instrument.subinstruments[i].primary_beam)
    if doplot:
        imed = n_nus // 2
        if idet is None:
            # idet = np.random.randint(Ndet)
            idet = 0
        sb = q_instrument.subinstruments[imed].get_synthbeam(q_scene, idet)  
        plt.figure()
        hp.gnomview(np.log10(sb/np.max(sb)), rot=[0,90], reso=20, min=-5, max=0,
             title='Theory {0:} {1:7.2f} GHz: TES #{2:}'.format(d['config'], q_instrument.subinstruments[imed].d['filter_nu']/1e9, idet), 
             sub=(1,2,1))
        for i in range(n_nus):
            hp.projscatter(thetas[i, idet,:], phis[i, idet,:], c=vals[i, idet,:]/np.max(vals[i, idet,:]), 
                           marker='x', cmap='Reds')
            # hp.projscatter(thetas[i, idet,:], phis[i, idet,:], c=vals[i, idet,:]/vals[i, idet,:], 
            #                marker='x', cmap='Reds')

        plt.subplot(1,2,2)
        plt.errorbar(nus, dnus, yerr=0, xerr=dnus/2, fmt='ro')
        for i in range(len(nus)):
            plt.axvline(x=nus[i]-dnus[i]/2, ls=':', color='k', alpha=0.5)
            plt.axvline(x=nus[i]+dnus[i]/2, ls=':', color='k', alpha=0.5)
        plt.xlabel('Frequency [GHz]')
        plt.ylabel('Bandwidth [GHz]')
        plt.title('Theory {0:} {1:7.2f} GHz: TES #{2:}'.format(d['config'], q_instrument.subinstruments[imed].d['filter_nu']/1e9, idet))
        plt.tight_layout()

    return thetas, phis, vals, nus, q_instrument

def update_dict(config, instrument_type, nf_sub, dictfilename='qubic/qubic/dicts/pipeline_demo.dict', debug=True):
    d_ = qubicDict()
    d_.read_from_file(dictfilename)
    d_['config'] = config
    
    d_['instrument_type'] = instrument_type
    d_['nf_sub'] = nf_sub
    d_['debug'] = debug
    d_['nside'] = d["nside"]              # To use the same as the maps

    d_['beam_shape'] = 'gaussian'  # can be 'gaussian', 'fitted_beam' or 'multi_freq'  
    d_['synthbeam'] = None         # we put nothing
    d_['use_synthbeam_fits_file'] = False
    d_['synthbeam_fraction'] = 1
    d_['synthbeam_kmax'] = 1
    return d_



In [ ]:
config = 'TD'
instrument_type = 'MB'
nf_sub = 12 # has to match d['nf_sub'] ?
d_ = update_dict(config, instrument_type, nf_sub)

idet = None #231
thetas, phis, vals, nus, qi = get_peaks_configuration(d_,  
                                                      doplot=True, 
                                                      idet=idet)

In [ ]:
print(np.shape(thetas))

thetas_det = thetas[:, 0, :] # radians, 90 - elevation
phis_det = phis[:, 0, :] # radians, azimuth
theo_sb = [thetas_det, phis_det]

## We first read a real data file to get the scanning strategy from it and choose the corresponding observation site

In [ ]:
mydatadir = '/Users/huchet/Documents/code/data/ComissioningTD/'
mydatadir3 = "/Users/huchet/Documents/code/data/"
thedate = '2022-07-14'
thedirs = glob.glob(mydatadir + '/' + thedate + '/*')
datadirs = [thedirs[0]]
azqubic = 116.4

# 2022 data
Salta = EarthLocation(lat=-24.731375*u.deg, lon=-65.409551*u.deg, height=1152*u.m)
####utcoffset = -3*u.hour  # Eastern Daylight Time

Obs_Site = Salta

# ### Read data
# a = qubicfp()
# a.read_qubicstudio_dataset(datadirs)

# ### We don;t need to fill the memory with the TODs
# #tt, alltod = a.tod()

# az = a.azimuth() + azqubic
# # az = -a.azimuth() + azqubic # testing a reverted azimuth --> not working, if the starting point's azimuth is well-known, then if we were going the wrong direction we wouldn't see the Moon!
# el = a.elevation()
# thk = a.timeaxis(datatype='hk')
# del a

In [ ]:
ObsSite = {'lat': Obs_Site.lat,
            'lon': Obs_Site.lon,
            'height': Obs_Site.height}

tshift = 0

data = pmf.format_data(az_qubic=azqubic, ObsSite=ObsSite,
                        speedmin=0.1, datadir=datadirs, tshift=tshift,
                        year_data=thedate[:4], simu=True)

tt, tinit, alltod, QPidx, azt, elt, newazt, newelt, scantype, Tbath, T1K, ifile = data

## Now we choose the instrument configuration (will probably be TD for a while)

This is to be updated with the current version of the soft

In [ ]:
### Let's simulate TOD from this
# Repository for dictionary and input maps
global_dir = Qubic_DataDir(datafile='instrument.py', datadir='../')
dictfilename = global_dir + '/dicts/pipeline_MoonSalta.dict'
d = Qdictionary.qubicDict()
d.read_from_file(dictfilename)
d['nside'] = 512 #256
d['config'] = 'TD' # no calfile for TD, or even for FI at 150 GHz
d['kind'] = 'I'
d['nf_sub'] = 12 # monoband not implemented on this branch
d['nf_recon'] = 2
d['MultiBand'] = True
d['sweeping_pointing_deadtime'] = False
d['noiseless'] = True
d["use_synthbeam_fits_file"] = False
d["n_nu_fit"] = d['nf_sub'] # monoband not implemented on this branch, but seems like all nsub are in the right band

d['detector_tau'] = 0.01

## We then create the simulated TOD for a given Moon spectrum

In [ ]:
# moon_spectrum_true = np.ones(d["n_nu_fit"])
# TOD_file = "simulated_TOD_flat" # moon_spectrum_true = np.ones(d["n_nu_fit"])

moon_spectrum_true = np.ones(d["n_nu_fit"]) * (np.arange(d["n_nu_fit"]) + 1)
TOD_file = "simulated_TOD_steep" # moon_spectrum_true = np.ones(d["n_nu_fit"]) * (np.arange(d["n_nu_fit"]) + 1)

# moon_spectrum_true = np.ones(d["n_nu_fit"]) * (np.arange(d["n_nu_fit"]) + 1)**4
# TOD_file = "simulated_TOD_steeper" # moon_spectrum_true = np.ones(d["n_nu_fit"]) * (np.arange(d["n_nu_fit"]) + 1)**4

We need to do it by batch in order not to overload the memory. It still takes a long time (>20 minutes)

In [ ]:
%%script echo we use saved TOD

size_batch = 100000 # pointings at a time
TOD_simu = []
for i_batch in range(len(tt)//size_batch + 1):
    print("\nBatch number {}".format(i_batch))
    range_min = i_batch*size_batch
    range_max = (i_batch + 1)*size_batch
    TOD_simu.append(simumoon.create_moon_tod(d, azt[range_min:range_max], elt[range_min:range_max], tt[range_min:range_max] + tinit, Obs_Site, moon_spectrum_true))
TOD_simu = np.concatenate(TOD_simu, axis=1)
pickle.dump( TOD_simu, open( mydatadir + TOD_file, "wb" ) )
print("Saved! {}".format(mydatadir + TOD_file))

In [ ]:
TOD_simu = pickle.load( open( mydatadir + TOD_file, "rb" ) )

## We finally analyse it with our pipeline

First, for one TES only

In [ ]:
TESNum = 96 #163 #96
indexQS = pmf.iQP2iQS(TESNum-1)
print(indexQS)

doplot = True
check_back_forth = True
det_pos = None
clean_tod = True
manual = False
ObsDate = thedate
new_method_clean = False
# theo_sb = None

tod = -TOD_simu[indexQS]*1e26
map_TES, mapscounts = pmf.make_coadded_maps_TES(tt, tod, azt, elt, scantype, newazt, newelt, ifile=ifile,
                                                    TES_number=TESNum, nside=d["nside"], 
                                                    doplot=doplot, check_back_forth=check_back_forth,
                                                    det_pos=det_pos, clean_tod=clean_tod, manual=manual,
                                                    ObsName=ObsDate, new_method_clean=new_method_clean, theo_sb=theo_sb)

In [ ]:
%%script echo skipping
# Simple consistency check with old code
date_obs = str(datetime.utcfromtimestamp(tinit))
alltimes = Time(date_obs) + tt*u.second #+ 3*u.minute

### Local coordinates
frame_obs = AltAz(obstime=alltimes, location=Obs_Site)

### Moon
moon_observed = get_body('Moon', alltimes, Obs_Site)

moonaltazs_Salta = moon_observed.transform_to(frame_obs)  
myazmoon = moonaltazs_Salta.az.value
myelmoon = moonaltazs_Salta.alt.value

# Map-making
newazt_ = (azt - myazmoon) * np.cos(np.radians(elt))
newelt_ = (elt - myelmoon)
mapsb, mapcount = pmf.healpix_map(newazt_[scantype != 0], newelt_[scantype != 0], -tod[scantype != 0], nside=d["nside"])

# Display Results
rot = [0, 0]
# rot = [-0.701, 7.459]
# rot = moon_fit_sim[iTES][0]
plt.figure()
hp.gnomview(mapsb, reso=10, rot=rot, title='Az/El wrt Moon FP centered [QP #{} ; QS #{}]'.format(TESNum, indexQS+1))
plt.show()

## We now fit the order 0 peak to get the position of the Moon for the detector

In [ ]:
allTESNum = np.arange(256) + 1

#### Offsets from Créidhe
quadrant = 3 # for TD, quadrant = 3; formerly "pointing_offsets_fixed_hole.pickle"
# import pickle
offsets = pickle.load( open( mydatadir3 + 'pointing_offsets_Q{}.pickle'.format(quadrant), 'rb') )
print(np.shape(offsets))

mytesn = np.array(allTESNum)
xycreidhe = offsets[mytesn -1 , :]
print(np.shape(xycreidhe))

######## Here we apply the inversion betwwen Az and El ##############
invert_azel = True # was due to an inversion in the numbering of TES, or was it? It doesn't fit well with the inversion
if invert_azel is True:
    xycreidhe = np.flip(xycreidhe, axis=1)
# print(xycreidhe[0])

invert_az = True # might be an error in the data taking (orientation east/west)?
if invert_az is True:
    xycreidhe[:, 0] = -xycreidhe[:, 0]
# print(xycreidhe[0])

invert_el = True # might be because we do el - elmoon in our data?
if invert_el is True:
    xycreidhe[:, 1] = -xycreidhe[:, 1]

In [ ]:
maynooth_cart = pmf.spherical2cartesian(1, xycreidhe[:, 0], xycreidhe[:, 1], coord="horizontal", axis="last") # changed from "first" to "last" in order to be compatible with comes later
rot_mat_zen = pmf.get_simple_rotation_matrix(axis="y", angle=np.radians(90)) # we rotate it to zenith
maynooth_zen_cart = np.einsum("li,ik->lk", maynooth_cart, rot_mat_zen)
maynooth_zen = pmf.cartesian2spherical(maynooth_zen_cart[:, 0], maynooth_zen_cart[:, 1], maynooth_zen_cart[:, 2], coord="horizontal", axis="last")[:, 1:]

In [ ]:
print(np.shape(mymap))

In [ ]:
import fitting as fit
xs = 201
reso = 3
idx = np.where(np.array(allTESNum)==indexQS)[0][0]

mymap = map_TES.copy()
myrotinit = np.array([0, 90, 0]) # zenith is better
# m, ijres, ijerr = pmf.fit_one_tes(mymap, xs, reso, rot=myrotinit, azelguess=None, verbose=False, renorm=True, doplot=True, distok=25)

# azelguess = maynooth_zen[idx]
# fitted_maynooth = np.load('fitted_maynooth.npy')
# azelguess = fitted_maynooth[TESNum - 1]
azelguess = None
resfit, mapxy, mapfit, extent, fig, ijres, ijerr = pmf.fit_one_tes(mymap, xs, reso, rot=myrotinit, return_images=True, renorm=True, doplot=True, azelguess=azelguess, verbose=True, distok=25)
# replace with MCMC? mught be overkill?
xy_TES = [ijres[1], ijres[0]]
errxy_TES = [ijerr[1], ijerr[0]]
plt.show()

In [ ]:
print(xy_TES)

In [ ]:
# we reconstruct the map, centered on the order 0 position fitted just before

idet_qp = TESNum - 1
# iTES = TESNum - 1
iTES = indexQS
idet_qs = iTES

fitted_maynooth = np.load('fitted_maynooth.npy')
print(fitted_maynooth[iTES])
print(xy_TES)

doplot = True
check_back_forth = True
det_pos = np.array(xy_TES)
# det_pos = fitted_maynooth
clean_tod = True
manual = False
ObsDate = thedate
new_method_clean = True
# theo_sb = None

# tod = -TOD_simu[indexQS]*1e26

data = pmf.format_data(az_qubic=azqubic, ObsSite=ObsSite,
                        speedmin=0.1, datadir=datadirs, tshift=tshift,
                        year_data=ObsDate[:4], det_pos=det_pos, simu=True)

tt, tinit, alltod, QPidx, azt, elt, newazt_, newelt_, scantype, Tbath, T1K, ifile = data

if det_pos is not None:
    if len(np.shape(newazt_)) == 1:
        print("case 1")
        newazt_corr = newazt_
        newelt_corr = newelt_
    elif np.shape(newazt_)[0] == 1:
        print("case 2")
        newazt_corr = newazt_[0]
        newelt_corr = newelt_[0]
        # det_pos_i = det_pos
    else:
        print("case 3")
        newazt_corr = newazt_[iTES]
        newelt_corr = newelt_[iTES]

# plt.figure()
# plt.plot(newazt)
# plt.show()

# plt.figure()
# plt.plot(elt)
# plt.plot(newelt)
# plt.show()
# # et

map_TES_corr, mapscounts = pmf.make_coadded_maps_TES(tt, tod, azt, elt, scantype, newazt_corr, newelt_corr, ifile=ifile,
                                                    TES_number=TESNum, nside=d["nside"], 
                                                    doplot=doplot, check_back_forth=check_back_forth,
                                                    det_pos=det_pos, clean_tod=clean_tod, manual=manual,
                                                    ObsName=ObsDate, new_method_clean=new_method_clean, theo_sb=theo_sb, simu=True)

## We will now fit the Moon spectrum

Ideas tested so far:
- least_squares: doesn't fit right, probably too many d.o.f.
- curve_fit: same
- emcee on each spectrum value: same, have to reduce the number of nus a lot for it to work
- emcee on linear spectrum: best approach yet, can keep all nus and fit only 2 params

emcee on linear spectrum works for a flat spectrum and is OK with non-flat linear spectrum.

In [ ]:
moon_angsize = np.radians(0.5) # radians
zenith = np.array([0, 90, 0])

In [ ]:
def get_sb_nus(q_scene, q_instrument, idet_qp, theta_max, Npix, reso):
    idet_qs = pmf.iQP2iQS(idet_qp)
    i_peak = 4 # order 0 peak in the case of Kmax = 1
    n_nus = len(q_instrument)
    sb_proj_nus = np.zeros((Npix, Npix, n_nus))
    for i_nu in range(n_nus):
        # we get the shape of the synthbeam
        sb_test = q_instrument[i_nu].get_synthbeam(q_scene, idet_qs, theta_max)
        sb_test_moon = hp.smoothing(sb_test, fwhm=moon_angsize)
        # we get the position of the peaks
        thetas, phis, _ = Qinstrument.QubicInstrument._peak_angles_unsorted(q_scene, q_instrument[i_nu].filter.nu,
            q_instrument[i_nu].detector.center, q_instrument[i_nu].synthbeam, q_instrument[i_nu].horn, q_instrument[i_nu].primary_beam)
        els_deg = 90 - np.degrees(thetas)
        azs_deg = np.degrees(phis)
        det_pos = (azs_deg[idet_qs, i_peak], els_deg[idet_qs, i_peak], -azs_deg[idet_qs, i_peak]) # we put the order 0 peak at the centre
        # we swap the axes to fit what we see in the data!!
        # might be the same thing as the detector line of sights computed by Maynooth
        # sb_proj_nus[:, :, i_nu] = np.swapaxes(hp.gnomview(sb_test_moon, rot=det_pos, reso=reso, xsize=Npix, return_projected_map=True, no_plot=True).data, 0, 1)
        # plan B, we first rotate at zenith and then project from there in order to have the same pixel shape
        test_rot = [det_pos[0], det_pos[1] - 90, det_pos[2]]
        # print(det_pos)
        # print(test_rot)
        # we rotate the sunthbeam from zenith to the observed detector position
        rotator = hp.rotator.Rotator(rot=test_rot, coord=None, inv=None, deg=True, eulertype='Y')
        sb_test_moon_rot = rotator.rotate_map_alms(sb_test_moon)
        sb_proj_nus[:, :, i_nu] = np.swapaxes(hp.gnomview(sb_test_moon_rot, rot=zenith, reso=reso, xsize=Npix, return_projected_map=True, no_plot=True).data, 0, 1)
        # plt.figure()
        # hp.gnomview(sb_test_moon, rot=zenith, reso=reso, xsize=Npix, return_projected_map=False, no_plot=False, sub=(1, 2, 1))
        # # hp.gnomview(sb_test_moon, rot=det_pos, reso=reso, xsize=Npix, return_projected_map=False, no_plot=False, sub=(1, 2, 2))
        # plt.show()
    return sb_proj_nus

In [ ]:
q_scene = QubicScene(d)
q_instrument = QubicMultibandInstrument(d)

In [ ]:

# Test on one TES

theta_max = 25

reso_test = 4 #2
Npix_test = 300 #600
X = np.arange(Npix_test)
Y = np.arange(Npix_test)
# XX, YY = np.meshgrid(X, Y, indexing="xy") # pixel units, just for the interpolation
sb_proj_nus = get_sb_nus(q_scene, q_instrument, idet_qp, theta_max, Npix_test, reso_test)
map_proj = hp.gnomview(map_TES_corr, rot=zenith, reso=reso_test, xsize=Npix_test, return_projected_map=True, no_plot=True)
no_UNSEEN_mask = ~map_proj.mask
# interpolator = LinearNDInterpolator(np.moveaxis([XX[no_UNSEEN_mask], YY[no_UNSEEN_mask]], 0, -1), map_proj[no_UNSEEN_mask])
# map_proj = interpolator(np.moveaxis([XX, YY], 0, -1))


# sb_proj_nus = get_sb_nus(q_scene, q_instrument, idet_qp, theta_max, Npix, reso)
# map_proj = allmaps_ok_proj[np.cumsum(visibly_ok_arr)[idet_qp] - 1]
map_proj[~np.isfinite(map_proj)] = 0
map_proj[map_proj == hp.UNSEEN] = 0
map_proj[map_proj.mask] = 0
# map_proj = map_proj/amp_TES * all_simu_amp
fig, axs = plt.subplots(1, 2, figsize = (15, 7))
axs[0].set_title("sb_proj_nus")
axs[0].imshow(np.mean(sb_proj_nus, axis=-1))
axs[1].set_title("map_proj")
axs[1].imshow(map_proj)
plt.show()

In [ ]:
# 2nd emcee approach

def fit_fun_moon_spectrum(lin_spectrum_params, sb_proj_nus, map_proj, nus): # I should probably choose one convention (iTES being first or last axis always)
    Npix = int(np.sqrt(len(map_proj)))
    # print(Npix, len(map_proj))
    # n_nus = len(sb_proj_nus)//Npix**2
    n_nus = len(nus)
    weights = lin_spectrum_params[0] * nus + lin_spectrum_params[1] # should be value of nus and not arange here
    map_proj = map_proj.reshape((Npix, Npix))
    sb_proj_nus = sb_proj_nus.reshape((Npix, Npix, n_nus))
    # the difference betweent he maps should be done only neaar the peaks, need to devise the mask for that and input it into the function
    # let's use the synthbeam values as mask and also a mask describing the pixels seen
    mask_seen = map_proj != 0
    # we stop caring about the centre to try to reduce the dof
    # XX, YY = np.meshgrid(np.arange(Npix), np.arange(Npix), indexing="xy")
    # mask_close_cent = np.sqrt((XX - Npix//2)**2 + (YY - Npix//2)**2) > 20 # 20 pixels might not be enough?
    tot_mask = mask_seen#*mask_close_cent
    # fig, axs = plt.subplots(1, 2)
    # axs[0].imshow(tot_mask*sb_proj_nus[:, :, 0])
    # axs[1].imshow(map_proj)
    # plt.show()
    # azet
    return -np.sum(np.abs(tot_mask*(map_proj - np.sum(weights[None, None, :] * sb_proj_nus, axis=-1))))

# def log_prob(spectrum_weights, others):
#     if np.any(spectrum_weights<0):
#         return -np.inf
#     sb_proj_nus, map_proj, nus = others
#     return np.log(fit_fun_moon_spectrum(spectrum_weights, sb_proj_nus, map_proj, nus)/)

# fit_fun_moon_spectrum(moon_spectrum_true, sb_proj_nus, map_proj.ravel(), nus)
# azef

# 2 dims, we fit a line
ndim, nwalkers = 2, 100
p0 = np.random.rand(nwalkers, ndim) * 10
print(p0, np.shape(p0))

# Set up the backend: it will save the chain
# Don't forget to clear it in case the file already exists
file_chain = "test_chain2.h5"
!rm {file_chain}
backend = emcee.backends.HDFBackend(file_chain)
backend.reset(nwalkers, ndim)

# Initialize the sampler
sampler = emcee.EnsembleSampler(nwalkers, ndim, fit_fun_moon_spectrum, args=[sb_proj_nus.ravel(), map_proj.ravel(), nus], backend=backend)
sampler.run_mcmc(p0, 5000)


In [ ]:
reader = emcee.backends.HDFBackend(file_chain)

tau = reader.get_autocorr_time()
burnin = int(2 * np.max(tau))
thin = int(0.5 * np.min(tau))
samples = reader.get_chain(discard=burnin, flat=True, thin=thin)
log_prob_samples = reader.get_log_prob(discard=burnin, flat=True, thin=thin)
# log_prior_samples = reader.get_blobs(discard=burnin, flat=True, thin=thin)

print("burn-in: {0}".format(burnin))
print("thin: {0}".format(thin))
print("flat chain shape: {0}".format(samples.shape))
print("flat log prob shape: {0}".format(log_prob_samples.shape))
# print("flat log prior shape: {0}".format(log_prior_samples.shape))

In [ ]:
flat_samples = sampler.get_chain(discard=100, thin=15, flat=True)
print(flat_samples.shape)

labels = ["a", "b"]
fitted_lin_spectrum_params = []
for i in range(ndim):
    mcmc = np.percentile(flat_samples[:, i], [16, 50, 84])
    q = np.diff(mcmc)
    fitted_lin_spectrum_params.append(mcmc[1])
    # txt = "\mathrm{{{3}}} = {0:.3f}_{{-{1:.3f}}}^{{{2:.3f}}}"
    # txt = txt.format(mcmc[1], q[0], q[1], labels[i])
    # print(txt)
    print("{} = {} + {} - {}".format(labels[i], mcmc[1], q[0], q[1]))
fitted_spectrum = fitted_lin_spectrum_params[0] * nus + fitted_lin_spectrum_params[1]

In [ ]:
fitted_map = no_UNSEEN_mask*np.sum(fitted_spectrum[None, None, :] * sb_proj_nus, axis=-1)

plt.figure()
plt.plot(nus, fitted_spectrum)
plt.plot(nus, moon_spectrum_true/30)
plt.yscale("log")
plt.show()

plt.figure()
plt.plot(nus, fitted_spectrum/moon_spectrum_true)
plt.yscale("log")
plt.show()


vmin, vmax = np.min(map_proj), np.max(map_proj)
fig, axs = plt.subplots(1, 3)
axs[0].imshow(map_proj, vmin=vmin, vmax=vmax)
axs[1].imshow(fitted_map, vmin=vmin, vmax=vmax)
axs[2].imshow(map_proj - fitted_map, vmin=vmin, vmax=vmax)
plt.show()